In [1]:
# This is necessary to recognize the modules
import os
import sys
from decimal import Decimal
import warnings

#warnings.filterwarnings("ignore")

root_path = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
sys.path.append(root_path)

In [2]:
from core.backtesting import BacktestingEngine

backtesting = BacktestingEngine(root_path=root_path, load_cached_data=True)

2025-04-10 20:14:04,609 - root - ERROR - Error writing configs: [Errno 2] No such file or directory: '/home/pascal/anaconda3/envs/quants-lab/lib/python3.10/site-packages/conf/conf_client.yml'
Traceback (most recent call last):
  File "/home/pascal/anaconda3/envs/quants-lab/lib/python3.10/site-packages/hummingbot/client/config/config_helpers.py", line 876, in save_to_yml
    with open(yml_path, "w", encoding="utf-8") as outfile:
FileNotFoundError: [Errno 2] No such file or directory: '/home/pascal/anaconda3/envs/quants-lab/lib/python3.10/site-packages/conf/conf_client.yml'


In [3]:
from hummingbot.strategy_v2.utils.distributions import Distributions
from controllers.market_making.pmm_simple import PMMSimpleConfig
from controllers.market_making.pz_mm import PZMMControllerConfig
from hummingbot.strategy_v2.executors.position_executor.data_types import TrailingStop
import datetime
from typing import List
from decimal import Decimal

# Controller configuration
connector_name = "binance_perpetual"
trading_pair = "WLD-USDT"
interval = "1m"
backtesting_resolution = "1m"
start = int(datetime.datetime(2025, 3, 30).timestamp())
end = int(datetime.datetime(2025, 3, 31).timestamp())
# TODO: Check for "1s" backtesting resolution, important for MM algo?

# General 
total_amount_quote: Decimal = 1000.0
cooldown_time = 60
executor_refresh_time = 60
sell_spreads = Distributions.arithmetic(1, 1, 0.5)
buy_spreads = Distributions.arithmetic(1, 1, 0.5)
buy_amounts_pct: List[Decimal] = [0.01]
sell_amounts_pct: List[Decimal] = [0.01]

# Indicator Values
# hma_very_slow: int = 50
hma_slow: int = 20
hma_fast: int = 10
hma_diff_ma: int = 9
# rsi_length: int = 9
stoch_rsi_smoothing: int = 3
stoch_rsi_length: int = 9
natr_length: int = 14

# Triple barrier
time_limit: int = 60*15
tp_natr_factor: Decimal = 0.75
sl_natr_factor: Decimal = 3.0
# ts_activation_natr_factor: Decimal = 1
# ts_delta_natr_factor: Decimal = 0.75


# Controller
config = PZMMControllerConfig(
    connector_name=connector_name,
    trading_pair=trading_pair,
    sell_spreads=sell_spreads,
    buy_spreads=buy_spreads,
    buy_amounts_pct=buy_amounts_pct,
    sell_amounts_pct=sell_amounts_pct,
    total_amount_quote=total_amount_quote,
    interval=interval,
    time_limit=time_limit,
    cooldown_time=cooldown_time, 
    executor_refresh_time=executor_refresh_time,
    # hma_very_slow = hma_very_slow,
    hma_slow = hma_slow,
    hma_fast = hma_fast,
    hma_diff_ma=hma_diff_ma,
    # rsi_length = rsi_length,
    stoch_rsi_smoothing = stoch_rsi_smoothing,
    stoch_rsi_length =stoch_rsi_length,
    natr_length = natr_length,
    tp_natr_factor = tp_natr_factor,
    sl_natr_factor = sl_natr_factor,
    # ts_activation_natr_factor=ts_activation_natr_factor,
    # ts_delta_natr_factor=ts_delta_natr_factor
)

In [4]:
# Running the backtesting this will output a backtesting result object that has built in methods to visualize the results
trade_cost = 0.001
backtesting_result = await backtesting.run_backtesting(config, start, end, backtesting_resolution, trade_cost=trade_cost)

2025-04-10 20:14:05,613 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7655750c9870>
2025-04-10 20:14:05,614 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x765616aa85e0>, 95925.326721185)])']
connector: <aiohttp.connector.TCPConnector object at 0x7655750c9840>


2025-04-10 20:14:17,400 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x765596786a10>
2025-04-10 20:14:17,401 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x765596751d80>, 95937.113509517)])']
connector: <aiohttp.connector.TCPConnector object at 0x765596787130>


In [5]:
# backtesting_result.controller_config.dict()

In [14]:
# Let's see what is inside the backtesting results
# print(backtesting_result.get_results_summary())
backtesting_result.get_backtesting_figure()

In [15]:
# 2. The executors dataframe: this is the dataframe that contains the information of the orders that were executed
# import pandas as pd

executors_df = backtesting_result.executors_df
# executors_df.head()

In [16]:
executors_df.head()

,id,timestamp,type,close_timestamp,close_type,status,config,net_pnl_pct,net_pnl_quote,cum_fees_quote,filled_amount_quote,is_active,is_trading,custom_info,controller_id,side
0,BuCffbvoAASavJ6SSraiP1r6uhxduYFnh9uoCiUjtJRA,1743289980,position_executor,1743289980,CloseType.STOP_LOSS,RunnableStatus.TERMINATED,{'id': 'BuCffbvoAASavJ6SSraiP1r6uhxduYFnh9uoCi...,-0.0010000000000000000208166817117216851329430...,-0.4995411860281763205549054873699788004159927...,0.49954118602817632055490548736997880041599273...,999.08237205635259670088998973369598388671875,False,False,"{'close_price': 0.7733, 'level_id': 'buy_0', '...",None,BUY
1,EcLSsfjdv9qHRSpb6gwVuMnfHH77C9622YCSBZq98anP,1743290040,position_executor,1743290040,CloseType.STOP_LOSS,RunnableStatus.TERMINATED,{'id': 'EcLSsfjdv9qHRSpb6gwVuMnfHH77C9622YCSBZ...,-0.0010000000000000000208166817117216851329430...,-0.4995411860281763760660567186278058215975761...,0.49954118602817637606605671862780582159757614...,999.0823720563527103877277113497257232666015625,False,False,"{'close_price': 0.7732, 'level_id': 'buy_0', '...",None,BUY
2,ApcbcaBxvPCSUDLbyDXd7XnNMjvpP4jUqVWsMM35i5Wv,1743289980,position_executor,1743290100,CloseType.EARLY_STOP,RunnableStatus.TERMINATED,{'id': 'ApcbcaBxvPCSUDLbyDXd7XnNMjvpP4jUqVWsMM...,0,0,0,0,False,False,"{'close_price': 0.774, 'level_id': 'sell_0', '...",None,SELL
3,faZWXAeFRHMZL1e2t9r9ueYB16p4wZBjRvpoHGYrHdY,1743290100,position_executor,1743290100,CloseType.STOP_LOSS,RunnableStatus.TERMINATED,{'id': 'faZWXAeFRHMZL1e2t9r9ueYB16p4wZBjRvpoHG...,-0.0010000000000000000208166817117216851329430...,-0.4995411860281763205549054873699788004159927...,0.49954118602817632055490548736997880041599273...,999.08237205635259670088998973369598388671875,False,False,"{'close_price': 0.774, 'level_id': 'buy_0', 's...",None,BUY
4,CejMrGUNk9Y7Fda2CeNmmGziUkZZbLwxsnEuiFfatJnD,1743290160,position_executor,1743290160,CloseType.STOP_LOSS,RunnableStatus.TERMINATED,{'id': 'CejMrGUNk9Y7Fda2CeNmmGziUkZZbLwxsnEuiF...,-0.0010000000000000000208166817117216851329430...,-0.4995411860281763205549054873699788004159927...,0.49954118602817632055490548736997880041599273...,999.08237205635259670088998973369598388671875,False,False,"{'close_price': 0.774, 'level_id': 'buy_0', 's...",None,BUY


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Grab processed data
candles_df = backtesting_result.processed_data

start_time = "2025-03-30 08:00"
end_time = "2025-03-30 09:00"

filtered_df = candles_df.loc[start_time:end_time]

# Create subplots layout
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    row_heights=[0.60, 0.15, 0.15, 0.1],
    vertical_spacing=0.02,
    subplot_titles=["Price + Indicators", "a", "b"]
)

# Row 1: Close price
fig.add_trace(go.Scatter(
    x=filtered_df.index, y=filtered_df["close"],
    line=dict(color='#FFFFFF', width=2), name='Close'), row=1, col=1)


fig.add_trace(go.Scatter(
    x=filtered_df.index,
    y=filtered_df["signal_multiplier"],
    mode='lines',
    line=dict(color='green', width=2),
    name='signal_multiplier',
    marker=dict(size=6)
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=filtered_df.index,
    y=filtered_df["reference_price"],
    mode='lines',
    line=dict(color='green', width=2),
    name='reference_price',
    marker=dict(size=6)
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=filtered_df.index,
    y=filtered_df["close"],
    mode='lines',
    line=dict(color='white', width=2),
    name='close',
    marker=dict(size=6)
), row=3, col=1)


fig.add_trace(go.Scatter(
    x=filtered_df.index,
    y=filtered_df["price_multiplier"],
    mode='lines',
    line=dict(color='yellow', width=2),
    name='price_multiplier',
    marker=dict(size=6)
), row=4, col=1)


fig.update_layout(
    height=700,
    title="Backtest",
    showlegend=True,
    template="plotly_dark"
)

fig.show()



In [10]:
filtered_df

,timestamp_bt,open_bt,high_bt,low_bt,close_bt,volume_bt,quote_asset_volume_bt,n_trades_bt,taker_buy_base_volume_bt,taker_buy_quote_volume_bt,...,close,volume,quote_asset_volume,n_trades,taker_buy_base_volume,taker_buy_quote_volume,spread_multiplier,price_multiplier,signal_multiplier,reference_price


### Backtesting Analysis

### Scatter of PNL per Trade
This bar chart illustrates the PNL for each individual trade. Positive PNLs are shown in green and negative PNLs in red, providing a clear view of profitable vs. unprofitable trades.


In [11]:
import plotly.express as px

# Create a new column for profitability
executors_df['profitable'] = executors_df['net_pnl_quote'] > 0

# Create the scatter plot
fig = px.scatter(
    executors_df,
    x="timestamp",
    y='net_pnl_quote',
    title='PNL per Trade',
    color='profitable',
    color_discrete_map={True: 'green', False: 'red'},
    labels={'timestamp': 'Timestamp', 'net_pnl_quote': 'Net PNL (Quote)'},
    hover_data=['filled_amount_quote', 'side']
)

# Customize the layout
fig.update_layout(
    xaxis_title="Timestamp",
    yaxis_title="Net PNL (Quote)",
    legend_title="Profitable",
    font=dict(size=12, color="white"),
    showlegend=False,
    plot_bgcolor='rgba(0,0,0,0.8)',  # Dark background
    paper_bgcolor='rgba(0,0,0,0.8)',  # Dark background for the entire plot area
    xaxis=dict(gridcolor="gray"),
    yaxis=dict(gridcolor="gray")
)

# Add a horizontal line at y=0 to clearly separate profits and losses
fig.add_hline(y=0, line_dash="dash", line_color="lightgray")

# Show the plot
fig.show()

### Histogram of PNL Distribution
The histogram displays the distribution of PNL values across all trades. It helps in understanding the frequency and range of profit and loss outcomes.


In [12]:
fig = px.histogram(executors_df, x='net_pnl_quote', title='PNL Distribution')
fig.show()
